# SDC — Trích xuất feature DHCP / DNS / mDNS / TLS từ CIC IoT Dataset 2022

Bước đầu tiên là viết extractor
đọc pcap và xuất ra bảng feature thô (1 row / event) cho từng nguồn:

| Nguồn | Field | Ghi chú |
|---|---|---|
| DHCP | hostname, vendor_class_id, param_req_list (option order), message type | Discover/Offer/Request/ACK/NAK |
| DNS | qry.name, qry.type, rcode | UDP/53, không tính mDNS |
| mDNS | qry.name, qry.type | UDP/5353 |
| TLS | SNI, ALPN, cipher suites, version | Từ ClientHello, có TCP reassembly và loại GREASE |

**Không dùng CICFlowMeter/tshark** (không có sẵn trong môi trường, không có quyền cài qua apt) — dùng `scapy`
(pip, không cần binary hệ thống) để đọc pcap và tự parse thủ công layer TLS record (scapy chưa hỗ trợ tốt
dissect ClientHello nhẹ mà không cần load thêm crypto).

Output: 3 bảng CSV thô (`dhcp_features.csv`, `dns_features.csv`, `tls_features.csv`) trong `Data/features/`,
mỗi row gắn kèm metadata thiết bị (category, device, scenario, capture file) để bước sau (coverage check,
labeling, train) dùng lại.

In [ ]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

## Thư viện

Hàm và hằng dùng chung, chép **nguyên văn** (không còn notebook `00_common`). Notebook khác có bản sao của cùng hàm — sửa ở đây thì sửa cả các bản kia cho khớp: bảng *Hàm có bản sao ở đâu* trong `README.md`. Phần riêng của notebook này bắt đầu sau mục này.

In [ ]:
# --- 1. Nền: đường dẫn, lược đồ, mã hoá ---
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
print("Gốc dự án:", ROOT)

In [ ]:
# --- Lược đồ dữ liệu — head, nhóm feature, chia tập ---
import pandas as pd
SOURCES = ["dhcp", "dns", "mdns", "tls"]

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
import hashlib
import re
import struct
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


try:  # Packet parsing is not needed by record-only production deployments.
    from scapy.all import BOOTP, DHCP, DNS, Ether, IP, IPv6, PcapReader, Raw, TCP, UDP
except ImportError:  # pragma: no cover - exercised only in minimal production images
    BOOTP = DHCP = DNS = Ether = IP = IPv6 = PcapReader = Raw = TCP = UDP = None
MAX_TLS_STREAM_BYTES = 256 * 1024

RAW_FIELDS = {
    "dhcp": ("opt55", "opt60"),
    "dns": ("qname",),
    "mdns": ("qname",),
    "tls": ("version", "ciphers", "alpn", "sni"),
}
# Cột cờ có-mặt nằm ở `sdc_schema` vì đó là lược đồ chung, không riêng của bước parse.
# Gác này bắt trường hợp hai nơi kể tên nguồn lệch nhau — im lặng thì `n_sources` sai.
assert tuple(RAW_FIELDS) == tuple(SOURCES), (
    f"RAW_FIELDS {tuple(RAW_FIELDS)} lệch SOURCES {tuple(SOURCES)}"
)

DHCP_MSG_TYPES = {
    1: "DISCOVER", 2: "OFFER", 3: "REQUEST", 4: "DECLINE",
    5: "ACK", 6: "NAK", 7: "RELEASE", 8: "INFORM",
}
DNS_QTYPES = {1: "A", 28: "AAAA", 12: "PTR", 16: "TXT", 33: "SRV",
              5: "CNAME", 255: "ANY"}
DNS_PORTS = (53, 5353)


def _require_scapy():
    if PcapReader is None:
        raise RuntimeError("Packet extraction requires Scapy; install it with 'pip install scapy'")


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
        return bool(missing) if isinstance(missing, (bool, np.bool_)) else False
    except (TypeError, ValueError):
        return False


def _text(value, *, lowercase=False):
    if _is_missing(value):
        return None
    if isinstance(value, bytes):
        value = value.decode(errors="replace")
    value = str(value).strip()
    if not value:
        return None
    return value.lower() if lowercase else value


def normalize_mac(value):
    """Return a lowercase colon-separated six-byte MAC, or ``None``."""
    if isinstance(value, bytes):
        raw = value[:6].hex()
    else:
        raw = re.sub(r"[^0-9a-f]", "", str(value or "").lower())[:12]
    if len(raw) != 12 or raw == "0" * 12:
        return None
    return ":".join(raw[index:index + 2] for index in range(0, 12, 2))


def packet_source_mac(pkt):
    _require_scapy()
    if not pkt.haslayer(Ether):
        return None
    return normalize_mac(pkt[Ether].src)


def bootp_client_mac(pkt):
    _require_scapy()
    if not pkt.haslayer(BOOTP):
        return None
    return normalize_mac(bytes(pkt[BOOTP].chaddr or b"")[:6])


def packet_from_device(pkt, device_mac):
    """Whether the Ethernet source belongs to the requested device."""
    if device_mac is None:
        return True
    return packet_source_mac(pkt) == normalize_mac(device_mac)


def dhcp_matches_device(pkt, device_mac):
    """DHCP may match by Ethernet source or by BOOTP ``chaddr`` after a relay."""
    if device_mac is None:
        return True
    wanted = normalize_mac(device_mac)
    return packet_source_mac(pkt) == wanted or bootp_client_mac(pkt) == wanted


def _dhcp_options_dict(dhcp_layer):
    return {option[0]: option[1] for option in dhcp_layer.options
            if isinstance(option, tuple) and len(option) >= 2}


def extract_dhcp(pkt, ts=None):
    """Extract DHCP client identity fields in canonical and training-compatible names."""
    _require_scapy()
    if not pkt.haslayer(DHCP):
        return None
    options = _dhcp_options_dict(pkt[DHCP])
    hostname = _text(options.get("hostname"), lowercase=True)
    # Preserve VCI bytes for compatibility with exact L1 keys. The text encoder
    # performs its own lowercase normalization for L2.
    vendor_class_id = _text(options.get("vendor_class_id"))

    request_list = options.get("param_req_list")
    if isinstance(request_list, bytes):
        request_list = list(request_list)
    if request_list is not None and not isinstance(request_list, (list, tuple)):
        request_list = [request_list]
    param_req_list = (",".join(str(int(option)) for option in request_list)
                      if request_list else None)
    if param_req_list is None and vendor_class_id is None:
        return None

    msg_type_code = options.get("message-type")
    try:
        msg_type_code = int(msg_type_code)
    except (TypeError, ValueError):
        pass
    timestamp = float(pkt.time) if ts is None and hasattr(pkt, "time") else ts
    client_mac = bootp_client_mac(pkt)
    return {
        "proto": "dhcp",
        "opt55": param_req_list,
        "opt60": vendor_class_id,
        "ts": timestamp,
        "src_mac": packet_source_mac(pkt),
        "client_mac": client_mac,
        "msg_type_code": msg_type_code,
        "msg_type": DHCP_MSG_TYPES.get(msg_type_code, msg_type_code),
        "hostname": hostname,
        "vendor_class_id": vendor_class_id,
        "param_req_list": param_req_list,
    }


def _network_addresses(pkt):
    if IP is not None and pkt.haslayer(IP):
        return str(pkt[IP].src), str(pkt[IP].dst)
    if IPv6 is not None and pkt.haslayer(IPv6):
        return str(pkt[IPv6].src), str(pkt[IPv6].dst)
    return None, None

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def extract_dns(pkt, ts=None):
    """Extract DNS and mDNS queries; responses are never feature evidence."""
    _require_scapy()
    if not pkt.haslayer(DNS):
        return []
    transport = pkt[UDP] if pkt.haslayer(UDP) else (pkt[TCP] if pkt.haslayer(TCP) else None)
    if transport is None:
        return []
    ports = (int(transport.sport), int(transport.dport))
    if not any(port in DNS_PORTS for port in ports):
        return []

    dns = pkt[DNS]
    if bool(dns.qr):
        return []
    is_mdns = 5353 in ports
    src_ip, dst_ip = _network_addresses(pkt)
    timestamp = float(pkt.time) if ts is None and hasattr(pkt, "time") else ts
    rows = []
    questions = list(dns.qd or []) if isinstance(dns.qd, (list, tuple)) else [dns.qd]
    for question in questions:
        if question is None:
            continue
        qname = _text(question.qname, lowercase=True)
        if not qname:
            continue
        qname = qname.rstrip(".")
        rows.append({
            "proto": "mdns" if is_mdns else "dns",
            "qname": qname,
            "ts": timestamp,
            "src_ip": src_ip,
            "dst_ip": dst_ip,
            "is_mdns": is_mdns,
            "is_response": False,
            "qry_name": qname,
            "qry_type": DNS_QTYPES.get(question.qtype, question.qtype),
            "rcode": None,
        })
    return rows


def is_grease(value):
    """RFC 8701 GREASE values have both bytes equal and a low nibble of 0xA."""
    try:
        value = int(value)
    except (TypeError, ValueError):
        return False
    return 0 <= value <= 0xFFFF and (value & 0x0F0F) == 0x0A0A


def parse_tls_client_hello(raw):
    """Parse a complete TLS ClientHello record and remove GREASE from its fingerprint."""
    try:
        raw = bytes(raw)
        if len(raw) < 9 or raw[0] != 0x16:
            return None
        record_length = struct.unpack("!H", raw[3:5])[0]
        if record_length > MAX_TLS_STREAM_BYTES or len(raw) < 5 + record_length:
            return None
        handshake = raw[5:5 + record_length]
        if len(handshake) < 4 or handshake[0] != 0x01:
            return None
        handshake_length = int.from_bytes(handshake[1:4], "big")
        if len(handshake) < 4 + handshake_length:
            return None
        body = handshake[4:4 + handshake_length]

        position = 0
        if len(body) < 35:
            return None
        version = body[position:position + 2].hex()
        position += 34  # version + random
        session_length = body[position]
        position += 1
        if position + session_length + 2 > len(body):
            return None
        position += session_length

        cipher_length = struct.unpack("!H", body[position:position + 2])[0]
        position += 2
        if cipher_length % 2 or position + cipher_length > len(body):
            return None
        cipher_suites = [struct.unpack("!H", body[offset:offset + 2])[0]
                         for offset in range(position, position + cipher_length, 2)]
        cipher_suites = [value for value in cipher_suites if not is_grease(value)]
        position += cipher_length

        if position >= len(body):
            return None
        compression_length = body[position]
        position += 1
        if position + compression_length > len(body):
            return None
        position += compression_length

        sni, alpn = None, []
        if position + 2 <= len(body):
            extensions_length = struct.unpack("!H", body[position:position + 2])[0]
            position += 2
            extensions_end = position + extensions_length
            if extensions_end > len(body):
                return None
            while position + 4 <= extensions_end:
                extension_type, extension_length = struct.unpack(
                    "!HH", body[position:position + 4]
                )
                position += 4
                if position + extension_length > extensions_end:
                    return None
                data = body[position:position + extension_length]
                position += extension_length

                if extension_type == 0 and len(data) >= 5:
                    names_length = struct.unpack("!H", data[:2])[0]
                    cursor = 2
                    names_end = min(len(data), 2 + names_length)
                    while cursor + 3 <= names_end:
                        name_type = data[cursor]
                        name_length = struct.unpack("!H", data[cursor + 1:cursor + 3])[0]
                        cursor += 3
                        if cursor + name_length > names_end:
                            break
                        if name_type == 0:
                            sni = _text(data[cursor:cursor + name_length], lowercase=True)
                            break
                        cursor += name_length
                elif extension_type == 16 and len(data) >= 2:
                    protocols_length = struct.unpack("!H", data[:2])[0]
                    cursor = 2
                    protocols_end = min(len(data), 2 + protocols_length)
                    while cursor < protocols_end:
                        protocol_length = data[cursor]
                        cursor += 1
                        if cursor + protocol_length > protocols_end:
                            break
                        protocol = _text(data[cursor:cursor + protocol_length], lowercase=True)
                        if protocol:
                            alpn.append(protocol)
                        cursor += protocol_length

        ciphers = ",".join(str(value) for value in cipher_suites)
        if not ciphers:
            return None
        alpn_value = ",".join(alpn) if alpn else None
        return {
            "proto": "tls",
            "version": version,
            "ciphers": ciphers,
            "alpn": alpn_value,
            "sni": sni,
            "tls_version": version,
            "cipher_suites": ciphers,
            "n_cipher_suites": len(cipher_suites),
        }
    except (IndexError, TypeError, ValueError, struct.error):
        return None


def _tls_packet_metadata(pkt, ts):
    src_ip, dst_ip = _network_addresses(pkt)
    return {
        "ts": float(pkt.time) if ts is None and hasattr(pkt, "time") else ts,
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "dst_port": int(pkt[TCP].dport),
    }

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
class TLSStreamReassembler:
    """Small bounded TCP reassembler for outbound TLS ClientHello records."""

    def __init__(self, max_stream_bytes=MAX_TLS_STREAM_BYTES):
        self.max_stream_bytes = max_stream_bytes
        self._segments = {}
        self._emitted = set()

    @staticmethod
    def _flow_key(pkt):
        src_ip, dst_ip = _network_addresses(pkt)
        if src_ip is None:
            src_ip, dst_ip = packet_source_mac(pkt), None
        tcp = pkt[TCP]
        return src_ip, dst_ip, int(tcp.sport), int(tcp.dport)

    @staticmethod
    def _assemble(segments):
        chunks = []
        cursor = None
        for sequence, payload in sorted(segments.items()):
            if cursor is None:
                cursor = sequence
            if sequence > cursor:
                break
            offset = max(0, cursor - sequence)
            if offset < len(payload):
                chunks.append(payload[offset:])
                cursor += len(payload) - offset
        return b"".join(chunks)

    def feed(self, pkt, ts=None):
        _require_scapy()
        if not (pkt.haslayer(TCP) and pkt.haslayer(Raw)):
            return []
        payload = bytes(pkt[Raw].load)
        if not payload:
            return []
        flow = self._flow_key(pkt)
        segments = self._segments.setdefault(flow, {})
        sequence = int(pkt[TCP].seq)
        if len(payload) > len(segments.get(sequence, b"")):
            segments[sequence] = payload
        if sum(map(len, segments.values())) > self.max_stream_bytes:
            self._segments.pop(flow, None)
            return []

        stream = self._assemble(segments)
        found = []
        offset = 0
        while offset + 5 <= len(stream):
            if stream[offset] != 0x16 or stream[offset + 1] != 0x03:
                offset += 1
                continue
            record_length = struct.unpack("!H", stream[offset + 3:offset + 5])[0]
            total = 5 + record_length
            if record_length > self.max_stream_bytes:
                offset += 1
                continue
            if offset + total > len(stream):
                break
            record = stream[offset:offset + total]
            parsed = parse_tls_client_hello(record)
            if parsed is not None:
                digest = hashlib.sha256(record).digest()
                marker = (flow, digest)
                if marker not in self._emitted:
                    self._emitted.add(marker)
                    parsed.update(_tls_packet_metadata(pkt, ts))
                    found.append(parsed)
            offset += total
        return found


def extract_tls(pkt, ts=None, reassembler=None):
    """Extract one ClientHello; pass a reassembler to support TCP segmentation."""
    _require_scapy()
    if not (pkt.haslayer(TCP) and pkt.haslayer(Raw)):
        return None
    if reassembler is not None:
        rows = reassembler.feed(pkt, ts)
        return rows[0] if rows else None
    parsed = parse_tls_client_hello(bytes(pkt[Raw].load))
    if parsed is not None:
        parsed.update(_tls_packet_metadata(pkt, ts))
    return parsed


def infer_device_mac(pcap_path):
    """Infer the most frequent DHCP client MAC for a single-device capture."""
    _require_scapy()
    candidates = Counter()
    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            row = extract_dhcp(pkt)
            if row and row["client_mac"]:
                candidates[row["client_mac"]] += 1
    return candidates.most_common(1)[0][0] if candidates else None

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

In [ ]:
import os
import struct
import glob
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from scapy.all import PcapReader, DHCP, BOOTP, DNS, DNSQR, UDP, TCP, IP, Raw

BASE_DIR = ROOT / "Data" / "1-Power"
OUT_DIR = ROOT / "Data" / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert BASE_DIR.exists(), f"Không tìm thấy {BASE_DIR.resolve()} — giải nén 1-Power.tar.gz trước"

## 1. Liệt kê capture files

Cấu trúc thư mục: `1-Power/<Category>/<Device>/POWER/<file>.pcap`. Bỏ qua file rác AppleDouble (`._*.pcap`,
sinh ra khi giải nén archive tạo trên macOS — không phải pcap thật).

In [ ]:
def iter_captures(base_dir: Path):
    """Yield (category, device, scenario, pcap_path) for every real capture file."""
    for pcap_path in sorted(base_dir.glob("*/*/*/*.pcap")):
        if pcap_path.name.startswith("._"):
            continue
        scenario_dir = pcap_path.parent          # e.g. POWER
        device_dir = scenario_dir.parent          # e.g. Amazon Echo Dot 1
        category_dir = device_dir.parent          # e.g. Audio
        yield category_dir.name, device_dir.name, scenario_dir.name, pcap_path

captures = list(iter_captures(BASE_DIR))
print(f"{len(captures)} capture file(s) hop le")
pd.DataFrame(captures, columns=["category", "device", "scenario", "path"]).head(10)

## 2. DHCP extractor

Field cần: `bootp.option.hostname`, `bootp.option.vendor_class_id`, `bootp.option.list` (thứ tự option —
Parameter Request List, option 55) + message type (Discover/Offer/Request/ACK/NAK).

## 3. DNS / mDNS extractor

Phân biệt DNS (UDP/53) và mDNS (UDP/5353) qua port. Xuất từng query record (`dns.qry.name`, `dns.qry.type`)
và `rcode` khi packet là response.

## 4. TLS ClientHello extractor

Dùng parser chung ở mục Thư viện, có TCP reassembly và loại GREASE trước khi tạo fingerprint.

## 5. Chạy extractor trên toàn bộ capture

Đọc từng pcap theo kiểu streaming (`PcapReader`) để tránh load nguyên file lớn (~270MB) vào RAM cùng lúc.
Mỗi row được gắn thêm metadata thiết bị (`category`, `device`, `scenario`, `capture_file`).

In [ ]:
def meta(category, device, scenario, pcap_path):
    return {
        "category": category,
        "device": device,
        "scenario": scenario,
        "capture_file": pcap_path.name,
        "session_id": f"{device}_{scenario}_{pcap_path.stem}",
    }

dhcp_rows, dns_rows, tls_rows = [], [], []
capture_summary = []

for category, device, scenario, pcap_path in tqdm(captures, desc="capture files"):
    m = meta(category, device, scenario, pcap_path)
    n_dhcp = n_dns = n_mdns = n_tls = 0
    device_mac = infer_device_mac(pcap_path)
    if device_mac is None:
        raise RuntimeError(f"Không suy được MAC thiết bị từ {pcap_path}")
    tls_stream = TLSStreamReassembler()

    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            ts = float(pkt.time)

            dhcp_row = extract_dhcp(pkt, ts)
            if dhcp_row and dhcp_matches_device(pkt, device_mac):
                dhcp_rows.append({**m, **dhcp_row})
                n_dhcp += 1

            if not packet_from_device(pkt, device_mac):
                continue

            for dns_row in extract_dns(pkt, ts):
                dns_rows.append({**m, **dns_row})
                if dns_row["is_mdns"]:
                    n_mdns += 1
                else:
                    n_dns += 1

            tls_row = extract_tls(pkt, ts, tls_stream)
            if tls_row:
                tls_rows.append({**m, **tls_row})
                n_tls += 1

    capture_summary.append({**m, "n_dhcp": n_dhcp, "n_dns": n_dns, "n_mdns": n_mdns, "n_tls": n_tls})

dhcp_df = pd.DataFrame(dhcp_rows)
dns_df = pd.DataFrame(dns_rows)
tls_df = pd.DataFrame(tls_rows)
summary_df = pd.DataFrame(capture_summary)

print("dhcp_df:", dhcp_df.shape, "dns_df:", dns_df.shape, "tls_df:", tls_df.shape)

## 6. Lưu output

Ghi 3 bảng feature thô + bảng summary (số event/nguồn mỗi capture — dùng làm input thô cho bước coverage
check ở mục 4 của kế hoạch) ra `Data/features/`.

In [ ]:
dhcp_df.to_csv(OUT_DIR / "dhcp_features.csv", index=False)
dns_df.to_csv(OUT_DIR / "dns_features.csv", index=False)
tls_df.to_csv(OUT_DIR / "tls_features.csv", index=False)
summary_df.to_csv(OUT_DIR / "capture_summary.csv", index=False)

print("Da luu vao", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.csv")):
    print(" -", f.name)

## 7. Kiểm tra nhanh (preview)

Xem nhanh số lượng event theo nguồn / category / device để có cảm nhận ban đầu về độ phủ trước khi làm
coverage check đầy đủ (mục 4 của kế hoạch).

In [ ]:
print("Capture co it nhat 1 nguon feature:")
has_any = summary_df.assign(
    has_dhcp=summary_df.n_dhcp > 0,
    has_dns=summary_df.n_dns > 0,
    has_mdns=summary_df.n_mdns > 0,
    has_tls=summary_df.n_tls > 0,
)
has_any["n_sources"] = has_any[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].sum(axis=1)
display(has_any.groupby("category")[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].mean().round(2))

In [ ]:
print("Phan bo so nguon feature co mat / capture:")
display(has_any["n_sources"].value_counts().sort_index())

## 8. Mở rộng: xử lý Idle dataset (toàn mạng, gắn nhãn qua MAC)

Khác với Power (mỗi thiết bị 1 file riêng), Idle là **1 file/đêm × 30 đêm**, chứa traffic của cả 40 thiết bị
trộn chung trên cùng mạng (xem `Data/2-Idle/Readme.txt`). Không còn metadata tên thiết bị theo path — gắn
nhãn bằng cách join **Ethernet source MAC** của từng packet với bảng ground-truth `Search/Device List.xlsx`
(40 thiết bị, đã xác nhận khớp 100% MAC khi kiểm tra sample).

Vì capture liên tục cả đêm (không có ranh giới "1 lần bật nguồn = 1 session" như Power), mỗi event được gắn
thêm `session_id = <device>_<ngày>_<giờ UTC>` (bucket theo giờ) để làm đơn vị session cho bước tổng hợp
feature/coverage check sau này — cùng tinh thần "1 row/session" của kế hoạch, áp dụng cho dữ liệu liên tục.

In [ ]:
import re
from datetime import datetime, timezone

IDLE_DIR = ROOT / "Data" / "2-Idle"

device_list_df = pd.read_excel(ROOT / "Search" / "Device List.xlsx")
device_list_df["Category"] = device_list_df["Category"].ffill()

mac_to_device = {}
mac_to_category = {}
for _, row in device_list_df.iterrows():
    mac = str(row["MAC Address"]).strip().lower()
    mac_to_device[mac] = row["Device Name"]
    mac_to_category[mac] = row["Category"]

known_macs = set(mac_to_device)
print(f"{len(known_macs)} thiet bi trong bang MAC ground-truth")

In [ ]:
def idle_date_from_filename(pcap_path: Path) -> str:
    m = re.match(r"(\d{4})_(\d{2})_(\d{2})_Idle", pcap_path.stem)
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

idle_files = sorted(IDLE_DIR.glob("*_Idle.pcap"))
print(f"{len(idle_files)} idle capture file(s)")
[(f.name, idle_date_from_filename(f)) for f in idle_files[:3]]

### Vòng lặp trích xuất Idle

Tái sử dụng nguyên `extract_dhcp` / `extract_dns` / `extract_tls` đã viết ở mục 2-4. Chỉ khác: lọc theo
`Ether.src` nằm trong `known_macs` trước khi trích xuất (vừa để gắn nhãn thiết bị đúng, vừa bỏ qua sớm phần
traffic không phải từ 1 trong 40 thiết bị — ví dụ router/AP — để đỡ tốn thời gian xử lý).

In [ ]:
from scapy.all import Ether

idle_dhcp_rows, idle_dns_rows, idle_tls_rows = [], [], []
idle_capture_summary = []

for pcap_path in tqdm(idle_files, desc="idle capture files"):
    date_str = idle_date_from_filename(pcap_path)
    per_device_counts = {}  # mac -> {n_dhcp, n_dns, n_mdns, n_tls}
    tls_streams = {}

    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            if not pkt.haslayer(Ether):
                continue
            ts = float(pkt.time)
            src_mac = packet_source_mac(pkt)
            dhcp_row = extract_dhcp(pkt, ts)
            dhcp_mac = dhcp_row.get("client_mac") if dhcp_row else None
            owner_mac = src_mac if src_mac in known_macs else dhcp_mac
            if owner_mac not in known_macs:
                continue

            hour = datetime.fromtimestamp(ts, tz=timezone.utc).hour
            device = mac_to_device[owner_mac]
            category = mac_to_category[owner_mac]
            m = {
                "category": category,
                "device": device,
                "scenario": "IDLE",
                "capture_file": pcap_path.name,
                "date": date_str,
                "session_id": f"{device}_{date_str}_{hour:02d}",
            }

            counts = per_device_counts.setdefault(
                owner_mac, {"n_dhcp": 0, "n_dns": 0, "n_mdns": 0, "n_tls": 0}
            )

            if dhcp_row and dhcp_matches_device(pkt, owner_mac):
                idle_dhcp_rows.append({**m, **dhcp_row})
                counts["n_dhcp"] += 1

            if src_mac != owner_mac:
                continue

            for dns_row in extract_dns(pkt, ts):
                idle_dns_rows.append({**m, **dns_row})
                if dns_row["is_mdns"]:
                    counts["n_mdns"] += 1
                else:
                    counts["n_dns"] += 1

            tls_stream = tls_streams.setdefault(owner_mac, TLSStreamReassembler())
            tls_row = extract_tls(pkt, ts, tls_stream)
            if tls_row:
                idle_tls_rows.append({**m, **tls_row})
                counts["n_tls"] += 1

    for mac, counts in per_device_counts.items():
        idle_capture_summary.append({
            "category": mac_to_category[mac],
            "device": mac_to_device[mac],
            "scenario": "IDLE",
            "capture_file": pcap_path.name,
            "date": date_str,
            **counts,
        })

idle_dhcp_df = pd.DataFrame(idle_dhcp_rows)
idle_dns_df = pd.DataFrame(idle_dns_rows)
idle_tls_df = pd.DataFrame(idle_tls_rows)
idle_summary_df = pd.DataFrame(idle_capture_summary)

print("idle_dhcp_df:", idle_dhcp_df.shape, "idle_dns_df:", idle_dns_df.shape, "idle_tls_df:", idle_tls_df.shape)

## 9. Lưu output Idle + gộp chung với Power

In [ ]:
idle_dhcp_df.to_csv(OUT_DIR / "dhcp_features_idle.csv", index=False)
idle_dns_df.to_csv(OUT_DIR / "dns_features_idle.csv", index=False)
idle_tls_df.to_csv(OUT_DIR / "tls_features_idle.csv", index=False)
idle_summary_df.to_csv(OUT_DIR / "capture_summary_idle.csv", index=False)

print("Da luu Idle features vao", OUT_DIR.resolve())
for f in ["dhcp_features_idle.csv", "dns_features_idle.csv", "tls_features_idle.csv", "capture_summary_idle.csv"]:
    print(" -", f)

In [ ]:
combined_dhcp_df = pd.concat([dhcp_df, idle_dhcp_df], ignore_index=True)
combined_dns_df = pd.concat([dns_df, idle_dns_df], ignore_index=True)
combined_tls_df = pd.concat([tls_df, idle_tls_df], ignore_index=True)
combined_summary_df = pd.concat([summary_df, idle_summary_df], ignore_index=True)

combined_dhcp_df.to_csv(OUT_DIR / "dhcp_features_all.csv", index=False)
combined_dns_df.to_csv(OUT_DIR / "dns_features_all.csv", index=False)
combined_tls_df.to_csv(OUT_DIR / "tls_features_all.csv", index=False)
combined_summary_df.to_csv(OUT_DIR / "capture_summary_all.csv", index=False)

print("combined_dhcp_df:", combined_dhcp_df.shape)
print("combined_dns_df:", combined_dns_df.shape)
print("combined_tls_df:", combined_tls_df.shape)
print("Da luu ban gop (Power+Idle) vao", OUT_DIR.resolve())

## 10. Kiểm tra nhanh coverage Idle (so với Power ở mục 7)

In [ ]:
idle_has_any = idle_summary_df.assign(
    has_dhcp=idle_summary_df.n_dhcp > 0,
    has_dns=idle_summary_df.n_dns > 0,
    has_mdns=idle_summary_df.n_mdns > 0,
    has_tls=idle_summary_df.n_tls > 0,
)
print("Idle - ty le (thiet-bi x ngay) co tung nguon, theo category:")
display(idle_has_any.groupby("category")[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].mean().round(2))

In [ ]:
print("Idle - so ngay quan sat duoc it nhat 1 event / thiet bi (top 10 nhieu nhat):")
display(idle_summary_df.groupby("device").size().sort_values(ascending=False).head(10))

print()
print("Idle - so ngay quan sat duoc it nhat 1 event / thiet bi (bottom 10 it nhat, co the la thiet bi im lang):")
display(idle_summary_df.groupby("device").size().sort_values().head(10))